In [ ]:
from pathlib import Path

def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit("ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = data_dir()

# # Exercise — Module 6: Clustering
#
# **Dataset (โปรดอ่านก่อนใช้งาน):**
# - *Air Quality* — De Vito, S., Massera, E., Piga, M., Martinotto, L., Di Francia, G. (2008). "On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario", *Sensors and Actuators B: Chemical*, 129(2), 750–757; *Air Quality* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C59K5F (CC BY 4.0) — `datasets/air_quality_uci.csv`
#
# ต่อจาก book.ipynb: เราหา operating regimes ของเซนเซอร์ก๊าซด้วย K-Means (k=4) เห็น PCA loadings และใช้ DBSCAN จับ noise ไปแล้ว — แบบฝึกหัดนี้ให้คุณตัดสินใจแทน: (A) ปรับ feature set กับจำนวน k, (B) ใช้ผลลัพธ์เป็นเครื่องมือตรวจความผิดปกติและอ่านโครงสร้างเซนเซอร์
#
# เซลล์ด้านล่าง parse ข้อมูลให้เหมือน book.ipynb แล้ว (สองคอลัมน์ตัวแปรกำกับเวลา `ts` ถูกเก็บไว้ให้ด้วย) — รันก่อนทำข้ออื่น

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

PALETTE = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"]

FEATURES = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)",
            "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH"]
SENSORS = FEATURES[:5]                      # เฉพาะ 5 ช่องเซนเซอร์

aq = pl.read_csv(DATA / "air_quality_uci.csv", separator=";", null_values="-200")
aq = aq.filter(pl.col("Date").is_not_null())
for c in ["CO(GT)", "C6H6(GT)", "T", "RH", "AH"]:
    aq = aq.with_columns(pl.col(c).str.replace(",", ".").cast(pl.Float64))
aq = aq.with_columns(
    (pl.col("Date") + " " + pl.col("Time")).str.to_datetime("%d/%m/%Y %H.%M.%S").alias("ts")
)
clean = aq.drop_nulls(FEATURES).with_row_index("row_id")
print(clean.shape)

X7 = clean.select(FEATURES).to_numpy().astype(float)
scaler = StandardScaler().fit(X7)
X7s = scaler.transform(X7)

# ค่าอ้างอิงจาก book.ipynb (7 คอลัมน์, k=4)
km7 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X7s)
sil7 = silhouette_score(X7s, km7.labels_)
print(f"7-feature, k=4 → silhouette = {sil7:.4f}")

# ## ข้อ A1 — ตัด T/RH ออก: cluster จาก 5 ช่องเซนเซอร์เท่านั้น
#
# โจทย์: บางคนแย้งว่า "regime ควรถูกกำหนดจากการตอบสนองของเซนเซอร์อย่างเดียว ไม่ใช่สภาพอากาศ"
#
# 1. scale ข้อมูล 5 ช่อง `SENSORS` แยกของมันเอง (`StandardScaler` fit ใหม่)
# 2. fit K-Means **k=4, n_init=10, random_state=42** แล้วหา mean silhouette บนข้อมูล 5 มิตินั้น
# 3. เทียบกับค่า 7-feature ที่พิมพ์ให้แล้วด้านบน — แล้วตอบใน markdown: silhouette ดีขึ้นหรือแย่ลง? และ **silhouette เพียงพอเป็นตัวตัดสินไหม** เมื่อเป้าหมายคือ "regime ที่ห้องปฏิบัติการใช้อธิบาย T/RH ได้"

In [ ]:
X5 = clean.select(SENSORS).to_numpy().astype(float)
X5s = ...
km5 = ...
lab5 = ...
sil5 = ...
print(f"5-feature, k=4 → silhouette = {sil5:.4f} (7-feature = {sil7:.4f})")

# **คำใบ้:** โครงสร้างเหมือนเซลล์อ้างอิงด้านบนทุกอย่าง แค่เปลี่ยน X — `X5s = StandardScaler().fit_transform(X5)`; silhouette คำนวณบนข้อมูล scaled ตัวเดียวกับที่ fit

# คำตอบข้อ A1:

# ## ข้อ A2 — k=3 หรือ k=5 จึงจะ "ถูก" กับงานสอบเทียบ
#
# เพื่อนร่วมชั้นคนหนึ่งบอกว่า k=2 ได้ silhouette สูงสุด (0.323) อีกคนบอกว่าข้อมูลจริงมีหลาย regime
#
# 1. fit K-Means บน `X7s` ที่ **k=3** และ **k=5** (n_init=10, random_state=42) พร้อม mean silhouette ของแต่ละค่า
# 2. แปลง centroid ของทั้งสองกลับหน่วยจริงด้วย scaler — ใช้ `scaler.inverse_transform(km.cluster_centers_)` (scaler ของ 7 คอลัมน์ถูก fit ในเซลล์เตรียมข้อมูลแล้ว แต่ K-Means ต้อง fit ใหม่ในข้อนี้)
# 3. ตอบใน markdown: จาก silhouette + การอ่าน centroid ว่าแต่ละ regime "ตั้งชื่อได้" แค่ไหน คุณจะเลือกกี่ regime เพื่อใช้เป็นแผนสอบเทียบแยกตามสภาพแวดล้อม เพราะอะไร

In [ ]:
km3 = ...
km5b = ...
for name, km in [("k=3", km3), ("k=5", km5b)]:
    sil = ...
    cent = ...
    print(f"--- {name}: silhouette = {sil:.4f}")
    print(pl.DataFrame(np.round(cent, 1), schema=FEATURES).with_row_index("cluster"))

# **คำใบ้:** `scaler.inverse_transform(km.cluster_centers_)` คืนเมทริกซ์ (k, 7) หน่วยดิบ — ใช้ `np.round(..., 1)` แล้วห่อด้วย `pl.DataFrame(..., schema=FEATURES)` ให้ตารางอ่านง่าย

# คำตอบข้อ A2:

# ## ข้อ B1 — ชั่วโมงนอก regime: ใช้ DBSCAN เป็นเครื่องตรวจ
#
# ห้องปฏิบัติการอยากได้ "รายชื่อชั่วโมงที่ควรตรวจซ้ำ" จากงานวิ่งปีของเครื่อง
#
# 1. fit DBSCAN บน `X7s` ด้วยพารามิเตอร์เดียวกับ book.ipynb (`eps=1.0, min_samples=10`)
# 2. นับ noise points (label = −1) และแถบเล็ก ๆ ที่แยกออกมา (label ≥ 0 แต่สมาชิกไม่ถึง 100 คน)
# 3. พล็อตค่า `T` กับ `RH` ของทั้งซีรีส์ตามเวลา (`clean["ts"]`) แล้วซ้อนจุด noise ด้วยสีแดง — ชั่วโมงผิดปกติกระจุกอยู่ช่วงไหน
# 4. ตอบใน markdown: ชั่วโมงที่ถูกตีว่า noise น่าจะมีสาเหตุจากอะไร (ดู T/RH ของจุดเหล่านั้นประกอบ)

In [ ]:
db = ...
lab = ...
n_noise = ...
print(f"noise = {n_noise} ชั่วโมง ({n_noise / len(lab) * 100:.2f}%)")

fig, axes = plt.subplots(2, 1, figsize=(11, 4.8), sharex=True)
# TODO: แถบบน T ตามเวลา (สีเทา), ซ้อนจุด noise สีแดง; แถบล่างเช่นเดียวกันกับ RH
...
fig.tight_layout()
plt.show()

# **คำใบ้:** `mask = lab == -1` → `axes[0].plot(clean["ts"], clean["T"], color="0.75", lw=0.8)` แล้วตามด้วย `axes[0].scatter(clean.filter(mask)["ts"], clean.filter(mask)["T"], s=12, color="#e76f51")`

# คำตอบข้อ B1:

# ## ข้อ B2 — อ่าน loadings: ช่องไหนขยับพร้อมกัน
#
# 1. fit PCA (ทุก component) บนข้อมูล 5 ช่องเซนเซอร์ scaled (`X5s` จากข้อ A1)
# 2. พิมพ์ explained variance ratio ของ PC1–PC2 และตาราง loadings ของ PC1 กับ PC2
# 3. พล็อตแท่ง loadings แนวนอนสองชุด (PC1, PC2) แบบที่ทำใน book.ipynb
# 4. ตอบใน markdown: จากเครื่องหมายกับขนาดของ loading — ช่องเซนเซอร์คู่ไหน "ขยับพร้อมกัน" และช่องไหน "สวนทาง" เสมอ? (ถ้าอยากยืนยัน ลองเช็กสหสัมพันธ์ของ 5 ช่องดิบ: `clean.select(SENSORS).corr()`)

In [ ]:
pca5 = ...
print("EV ratio:", ...)
print(pl.DataFrame(np.round(pca5.components_[:2], 3), schema=SENSORS).with_columns(pl.Series("PC", ["PC1", "PC2"])))

fig, ax = plt.subplots(figsize=(8.5, 3.4))
# TODO: barh สองชุด PC1/PC2 แบบ book.ipynb
...
plt.show()

# **คำใบ้:** หลัง fit แล้ว `pca5.components_` เป็นเมทริกซ์ (5, 5) — แถวคือ PC; ใช้ `np.arange(5) - 0.19` กับ `+ 0.19` เป็นตำแหน่ง barh ให้แท่งไม่ทับกัน

# คำตอบข้อ B2:

# ## ข้อ B3 — โจทย์คิด (ตอบใน markdown)
#
# หัวหน้าห้องปฏิบัติการเสนอ: "งบมีจำกัด จึงจะสอบเทียบโมเดลการตอบสนองของเซนเซอร์แบบ **regime เดียวเท่านั้น** — เอา clustering มาช่วยตัดสินว่าจะยึดสภาพแวดล้อมแบบไหน"
#
# 1. จากผลของคุณในข้อ A1–A2 — ถ้าต้องเลือก **regime เดียว** สำหรับทำใบรับรอง ควรเลือกช่วง T/RH แบบไหน และข้อจำกัดที่ต้องเขียนกำกับใบรับรองคืออะไร (นึกถึงบทเรียน extrapolation จาก Module 3)
# 2. ก้อนเล็ก 17 ชั่วโมงของ DBSCAN (คืนหมอก 1–2 ธ.ค. 2004) กับ noise อีก ~0.6% — ทีมควร (ก) ลบทิ้ง (ข) รวมเข้า regime ใกล้สุด หรือ (ค) เก็บไว้ตรวจเป็นกรณีพิเศษ? ให้เหตุผลเชิงเมโทรโลยี